# Allstate Claims Severity
## 01 — Data Understanding

### Objective
The goal of this project is to predict insurance claim severity
using historical claim data.

This is a supervised regression problem where `loss` represents
the target variable.

### In This Notebook
- Load and inspect the dataset.
- Explore feature types and dataset structure.
- Identify missing values and duplicate records.
- Examine the distribution of the target variable.
- Summarize initial observations and findings.

In [ ]:
import sys

import numpy as np
import pandas as pd
import sklearn

print("Python path:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

## 1. Data Loading

Load the training and test datasets using Pandas.

The training dataset includes the target variable (`loss`),
while the test dataset is used for generating predictions.

We will first verify their dimensions and structure.

In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

## 2. Initial Data Inspection

Before preprocessing or modeling, we need to understand the structure
and characteristics of the datasets.

In this section, we will:
- Preview the first few records.
- Examine column names and data types.
- Identify the target variable and feature groups.
- Compare the training and test dataset structures.

In [ ]:
display(train.head())
display(test.head())

In [ ]:
train.info()

## 3. Data Quality Assessment

Before exploring feature distributions, we assess the basic
quality and integrity of the datasets.

This section focuses on:
- Detecting missing values in training and test data.
- Identifying duplicate records.
- Verifying the uniqueness of record identifiers.

The findings will help determine whether data cleaning is required.

In [ ]:
print("Train missing values:", train.isna().sum().sum())
print("Test missing values:", test.isna().sum().sum())

print("Train duplicates:", train.duplicated().sum())
print("Test duplicates:", test.duplicated().sum())

print("Train unique IDs:", train["id"].nunique())
print("Test unique IDs:", test["id"].nunique())

### Data Quality Findings

The initial data quality assessment shows that:

- Neither dataset contains missing values.
- No fully duplicated records were detected.
- No missing-value imputation or duplicate removal is required at this stage.

Record identifier uniqueness is checked separately to validate data integrity.

In [ ]:
train_duplicates = train.drop(columns=["id"]).duplicated().sum()
test_duplicates = test.drop(columns=["id"]).duplicated().sum()

print("Train duplicates (excluding ID):", train_duplicates)
print("Test duplicates (excluding ID):", test_duplicates)

In [ ]:
print("Train IDs unique:", train["id"].is_unique)
print("Test IDs unique:", test["id"].is_unique)

### Data Quality Findings

The initial data quality checks reveal the following:

- **Missing Values:** Neither dataset contains missing values.
- **Duplicate Records:** No fully duplicated records were detected, even after excluding the `id` column.
- **ID Uniqueness:** All record identifiers are unique within each dataset.
- **Data Cleaning:** No missing-value imputation or duplicate removal is required at this stage.

These results indicate that the datasets have good initial data quality.
Further analysis is needed to investigate outliers, feature distributions,
and potential inconsistencies.

## 4. Target Variable Analysis

The target variable `loss` represents the insurance claim amount
that our model aims to predict.

We will examine its statistical properties and distribution to
understand typical claim amounts, variability, and potential outliers.

These findings will guide our preprocessing and modeling decisions.

In [ ]:
train["loss"].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

### Initial Target Insights

The descriptive statistics suggest a positively skewed target distribution.

- The mean (3,037.34) exceeds the median (2,115.57), suggesting that large claims influence the average.
- The 99th percentile is approximately 13,981, while the maximum reaches 121,012.
- The large upper-tail values indicate potential extreme observations that require further investigation.

We will examine the distribution visually before making any preprocessing decisions.

### 4.1 Loss Distribution

We visualize the distribution of claim losses using a histogram.

This helps identify the most frequent loss ranges, assess distribution
asymmetry, and investigate the influence of unusually large claims.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

sns.histplot(data=train, x="loss", bins=60)

plt.title("Distribution of Insurance Claim Losses")
plt.xlabel("Claim Loss")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

### 4.2 Zoomed Loss Distribution

The full histogram is heavily compressed due to extreme claim values.

To better understand the distribution of typical claims, we visualize
loss values up to the 99th percentile.

This visualization excludes extreme values from the plot only;
the original dataset remains unchanged.

In [ ]:
p99 = train["loss"].quantile(0.99)

loss_zoomed = train.loc[train["loss"] <= p99, "loss"]

plt.figure(figsize=(10, 5))

sns.histplot(loss_zoomed, bins=60)

plt.title("Claim Loss Distribution (Up to 99th Percentile)")
plt.xlabel("Claim Loss")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
print("Loss skewness:", train["loss"].skew())

### 4.3 Boxplot and Outlier Detection

We use a boxplot and the Interquartile Range (IQR) method to
investigate unusually large insurance claims.

The analysis aims to:
- Examine the spread of the middle 50% of claim losses.
- Identify observations outside the standard IQR boundaries.
- Assess whether extreme values require further investigation.

Statistical outliers will not be automatically removed.

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(x=train["loss"])

plt.title("Boxplot of Insurance Claim Losses")
plt.xlabel("Claim Loss")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(x=train["loss"])

plt.xlim(0, train["loss"].quantile(0.99))

plt.title("Claim Loss Boxplot (Zoomed to 99th Percentile)")
plt.xlabel("Claim Loss")

plt.tight_layout()
plt.show()

In [ ]:
q1 = train["loss"].quantile(0.25)
q3 = train["loss"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Lower bound: ", lower_bound)
print("Upper bound: ", upper_bound)

In [ ]:
outlier_mask = (train["loss"] < lower_bound) | (train["loss"] > upper_bound)

In [ ]:
outlier_mask.sum()

In [ ]:
outlier_mask.mean() * 100

### 4.4 Impact of High-Loss Claims

The IQR method identifies approximately 6.14% of claims as
statistical outliers.

Rather than removing these observations, we investigate their
contribution to the total claim amount and compare their
characteristics with the remaining claims.

This helps assess their business significance before making
any modeling decisions.

In [ ]:
regular_losses = train.loc[~outlier_mask, "loss"]
high_losses = train.loc[outlier_mask, "loss"]

print("Regular claims mean:", regular_losses.mean())
print("High-loss claims mean:", high_losses.mean())

total_loss = train["loss"].sum()
high_loss_total = high_losses.sum()

high_loss_share = high_loss_total / total_loss * 100

print(f"High-loss share of total losses: {high_loss_share:.2f}%")

ratio = high_losses.mean() / regular_losses.mean()

print(f"High-loss claims are {ratio:.2f} times larger on average.")

### 4.5 Target Analysis Findings

The target variable `loss` exhibits a strongly right-skewed distribution,
with a skewness of approximately 3.79. The mean exceeds the median,
indicating the influence of large insurance claims.

The IQR method identifies 11,554 observations (6.14%) as statistical
outliers. These claims account for approximately 22.91% of the total
claim amount and have an average loss 4.55 times that of regular claims.

**Modeling Implications:**
- Retain extreme claims unless evidence suggests data errors.
- Establish a baseline before applying target transformations.
- Evaluate prediction errors across both regular and high-loss claims.
- Compare any transformations using MAE on the original loss scale.